<a href="https://colab.research.google.com/github/Halidh-Ahamed/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [27]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Refresh pages with 100 or more Google Search impressions and an average search position between 5 and 35. These pages already have meaningful visibility and are in a ranking range where content improvements could realistically increase traffic. Pages that satisfy only one condition are monitored or reviewed, while pages satisfying neither are ignored.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [28]:
feature_df = con.sql(f"""
SELECT
    report_date,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    client_has_gsc,
    client_has_ga4,
    gsc_data_available
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
""").df()

feature_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,client_has_gsc,client_has_ga4,gsc_data_available
0,2026-03-01,content_b7e512995f79d5a6,20,0,67,True,False,True
1,2026-03-01,content_05597932fe4da067,1,0,0,True,False,True
2,2026-03-01,content_7a105f548d9c6916,125,1,616,True,False,True
3,2026-03-01,content_905aa32a0230694e,7,0,28,True,False,True
4,2026-03-01,content_a3ea9792f793ec72,11,0,25,True,False,True


In [29]:
feature_df["avg_position"] = (
    feature_df["gsc_sum_position"] /
    feature_df["gsc_impressions"].replace(0, 1)
)

feature_df["ctr"] = (
    feature_df["gsc_clicks"] /
    feature_df["gsc_impressions"].replace(0, 1)
)

feature_df.head()

,report_date,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,client_has_gsc,client_has_ga4,gsc_data_available,avg_position,ctr
0,2026-03-01,content_b7e512995f79d5a6,20,0,67,True,False,True,3.350000,0.000
1,2026-03-01,content_05597932fe4da067,1,0,0,True,False,True,0.000000,0.000
2,2026-03-01,content_7a105f548d9c6916,125,1,616,True,False,True,4.928000,0.008
3,2026-03-01,content_905aa32a0230694e,7,0,28,True,False,True,4.000000,0.000
4,2026-03-01,content_a3ea9792f793ec72,11,0,25,True,False,True,2.272727,0.000


In [30]:
feature_df["action_score"] = (
    (feature_df["gsc_impressions"] >= 100).astype(int) +
    (
        (feature_df["avg_position"] >= 5) &
        (feature_df["avg_position"] <= 35)
    ).astype(int)
)

feature_df[["gsc_impressions", "avg_position", "action_score"]].head()

,gsc_impressions,avg_position,action_score
0,20,3.350000,0
1,1,0.000000,0
2,125,4.928000,1
3,7,4.000000,0
4,11,2.272727,0


In [31]:
def get_reason(score):
    if score == 2:
        return "HIGH_VOLUME_GOOD_POSITION"
    elif score == 1:
        return "PARTIAL_SIGNAL"
    else:
        return "LOW_SIGNAL"

feature_df["reason_code"] = feature_df["action_score"].apply(get_reason)

feature_df[
    ["gsc_impressions", "avg_position", "action_score", "reason_code"]
].head()

,gsc_impressions,avg_position,action_score,reason_code
0,20,3.350000,0,LOW_SIGNAL
1,1,0.000000,0,LOW_SIGNAL
2,125,4.928000,1,PARTIAL_SIGNAL
3,7,4.000000,0,LOW_SIGNAL
4,11,2.272727,0,LOW_SIGNAL


In [32]:
def get_action(score):
    if score == 2:
        return "REFRESH"
    elif score == 1:
        return "MONITOR"
    else:
        return "IGNORE"

feature_df["action"] = feature_df["action_score"].apply(get_action)

feature_df[
    [
        "gsc_impressions",
        "avg_position",
        "action_score",
        "reason_code",
        "action"
    ]
].head()

,gsc_impressions,avg_position,action_score,reason_code,action
0,20,3.350000,0,LOW_SIGNAL,IGNORE
1,1,0.000000,0,LOW_SIGNAL,IGNORE
2,125,4.928000,1,PARTIAL_SIGNAL,MONITOR
3,7,4.000000,0,LOW_SIGNAL,IGNORE
4,11,2.272727,0,LOW_SIGNAL,IGNORE


In [25]:
import os

# Sort the pages by priority
feature_df = feature_df.sort_values(
    by=["action_score", "gsc_impressions"],
    ascending=[False, False]
).reset_index(drop=True)

# Create ranking
feature_df["rank"] = feature_df.index + 1

# Create output folder
os.makedirs("work/outputs", exist_ok=True)

# Save CSV
feature_df.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

# Preview top 20 pages
feature_df.head(20)

,report_date,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,client_has_gsc,client_has_ga4,gsc_data_available,avg_position,ctr,action_score,reason_code,action,rank
0,2026-03-04,content_945d6ff91386c817,37368,0,321886,True,False,True,8.613948,0.000000,2,HIGH_VOLUME_GOOD_POSITION,REFRESH,1
1,2026-03-29,content_66288edeb93b7c4f,24577,66,265290,True,True,True,10.794239,0.002685,2,HIGH_VOLUME_GOOD_POSITION,REFRESH,2
2,2026-03-28,content_66288edeb93b7c4f,23542,165,261602,True,True,True,11.112140,0.007009,2,HIGH_VOLUME_GOOD_POSITION,REFRESH,3
3,2026-03-15,content_1642f339bd6e7c8d,16454,1,84435,True,False,True,5.131579,0.000061,2,HIGH_VOLUME_GOOD_POSITION,REFRESH,4
4,2026-03-28,content_e943d753806d7af3,15522,49,136414,True,True,True,8.788429,0.003157,2,HIGH_VOLUME_GOOD_POSITION,REFRESH,5
5,2026-03-09,content_e8a52cf3d5988c07,15394,45,266267,True,True,True,17.296804,0.002923,2,HIGH_VOLUME_GOOD_POSITION,REFRESH,6
6,2026-03-31,content_e6df0936699f5b8f,14682,269,367576,True,True,True,25.035826,0.018322,2,HIGH_VOLUME_GOOD_POSITION,REFRESH,7
7,2026-03-28,content_046fc480045b88f5,14185,1,100009,True,True,True,7.050335,0.000070,2,HIGH_VOLUME_GOOD_POSITION,REFRESH,8
8,2026-03-12,content_e8a52cf3d5988c07,14138,35,229662,True,True,True,16.244306,0.002476,2,HIGH_VOLUME_GOOD_POSITION,REFRESH,9
9,2026-03-11,content_e8a52cf3d5988c07,13910,30,232025,True,True,True,16.680446,0.002157,2,HIGH_VOLUME_GOOD_POSITION,REFRESH,10


In [26]:
feature_df[
    [
        "rank",
        "content_hash_id",
        "gsc_impressions",
        "avg_position",
        "ctr",
        "action",
        "reason_code"
    ]
].head(20)

,rank,content_hash_id,gsc_impressions,avg_position,ctr,action,reason_code
0,1,content_945d6ff91386c817,37368,8.613948,0.000000,REFRESH,HIGH_VOLUME_GOOD_POSITION
1,2,content_66288edeb93b7c4f,24577,10.794239,0.002685,REFRESH,HIGH_VOLUME_GOOD_POSITION
2,3,content_66288edeb93b7c4f,23542,11.112140,0.007009,REFRESH,HIGH_VOLUME_GOOD_POSITION
3,4,content_1642f339bd6e7c8d,16454,5.131579,0.000061,REFRESH,HIGH_VOLUME_GOOD_POSITION
4,5,content_e943d753806d7af3,15522,8.788429,0.003157,REFRESH,HIGH_VOLUME_GOOD_POSITION
5,6,content_e8a52cf3d5988c07,15394,17.296804,0.002923,REFRESH,HIGH_VOLUME_GOOD_POSITION
6,7,content_e6df0936699f5b8f,14682,25.035826,0.018322,REFRESH,HIGH_VOLUME_GOOD_POSITION
7,8,content_046fc480045b88f5,14185,7.050335,0.000070,REFRESH,HIGH_VOLUME_GOOD_POSITION
8,9,content_e8a52cf3d5988c07,14138,16.244306,0.002476,REFRESH,HIGH_VOLUME_GOOD_POSITION
9,10,content_e8a52cf3d5988c07,13910,16.680446,0.002157,REFRESH,HIGH_VOLUME_GOOD_POSITION


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

| Rank | Action | Reason Code | Confidence Note | What would make it wrong? |
|------|--------|----------------------------|------------------------------------------------------|--------------------------------------------------------------|
| 1 | REFRESH | HIGH_VOLUME_GOOD_POSITION | High confidence because the page has very high impressions and ranks within the target range. | The page may already be optimized or seasonal, so refreshing may not improve performance. |
| 2 | REFRESH | HIGH_VOLUME_GOOD_POSITION | High confidence because it has strong visibility and room for ranking improvement. | Search demand could have dropped or rankings may already be stable. |
| 3 | REFRESH | HIGH_VOLUME_GOOD_POSITION | High confidence because it satisfies both rule conditions. | External factors such as algorithm updates may limit improvement. |
| 4 | REFRESH | HIGH_VOLUME_GOOD_POSITION | High confidence because it receives substantial impressions and has moderate rankings. | The content may not actually require updates despite matching the rule. |
| 5 | REFRESH | HIGH_VOLUME_GOOD_POSITION | High confidence because it has high search exposure. | Poor user intent match could prevent gains after refreshing. |
| 6 | REFRESH | HIGH_VOLUME_GOOD_POSITION | High confidence because both decision criteria are satisfied. | Ranking changes may be caused by competition rather than outdated content. |
| 7 | REFRESH | HIGH_VOLUME_GOOD_POSITION | High confidence because the page has meaningful visibility. | The page could already be performing as expected for its topic. |
| 8 | REFRESH | HIGH_VOLUME_GOOD_POSITION | High confidence because the page has strong impressions within the selected ranking range. | Technical SEO issues rather than content quality may be limiting performance. |
| 9 | REFRESH | HIGH_VOLUME_GOOD_POSITION | High confidence because it matches the baseline rule completely. | Seasonal search trends may make refreshing ineffective. |
| 10 | REFRESH | HIGH_VOLUME_GOOD_POSITION | High confidence because it has significant search visibility. | The page may already have reached its ranking potential. |
| 11 | REFRESH | HIGH_VOLUME_GOOD_POSITION | High confidence because both thresholds are satisfied. | Search intent may have changed since the content was published. |
| 12 | REFRESH | HIGH_VOLUME_GOOD_POSITION | High confidence because the page qualifies on all rule conditions. | The issue may not be content freshness but backlinks or authority. |
| 13 | REFRESH | HIGH_VOLUME_GOOD_POSITION | High confidence because impressions indicate meaningful traffic opportunity. | Refreshing could have little effect if rankings are already stable. |
| 14 | REFRESH | HIGH_VOLUME_GOOD_POSITION | High confidence because the page falls inside the target position range. | CTR improvements rather than content updates may be needed. |
| 15 | REFRESH | HIGH_VOLUME_GOOD_POSITION | High confidence because the page has high impressions. | The page may already satisfy user needs well enough. |
| 16 | REFRESH | HIGH_VOLUME_GOOD_POSITION | High confidence because both baseline conditions are met. | Declining performance may be caused by external market changes. |
| 17 | REFRESH | HIGH_VOLUME_GOOD_POSITION | High confidence because the page has good visibility and improvement potential. | Search volume could naturally fluctuate over time. |
| 18 | REFRESH | HIGH_VOLUME_GOOD_POSITION | High confidence because the page matches the refresh criteria. | Ranking losses may be unrelated to content quality. |
| 19 | REFRESH | HIGH_VOLUME_GOOD_POSITION | High confidence because impressions remain consistently high. | Technical issues or indexing problems could be the true cause. |
| 20 | REFRESH | HIGH_VOLUME_GOOD_POSITION | High confidence because the page satisfies every baseline condition. | The rule does not consider seasonality, competition, or content quality directly. |


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak picks + leakage check

Some selected pages may still be weak candidates for refresh. For example, pages with high impressions may already be performing close to their maximum potential, or their rankings may be limited by competition, search intent, seasonality, or technical SEO issues rather than outdated content. Because this baseline rule only considers impressions and average position, it cannot distinguish these situations.

I also confirmed that no data leakage occurred. The rule uses only information available during March 2026 (Google Search impressions and average position). It does not use future performance, product flags, outcome labels, or any information from later time windows. Therefore, the ranking is based only on information that would have been available at the time the decision was made.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.